# 06 · Conjunto de prueba final

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Construye la prueba de 10.000 comentarios con la que se evalúan los 18 modelos: solo comentarios del etiquetado masivo que no están en entrenamiento ni en validación, sin textos repetidos (ignorando mayúsculas y espacios). Verifica estas condiciones antes de guardar.

*Nombre durante el desarrollo: `04c_Prueba_Final` (algunas salidas conservan esa numeración).*

## 1. Configuración

In [1]:
!pip install -q -U huggingface_hub scikit-learn pandas

import os, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import userdata, files
from huggingface_hub import login, hf_hub_download, upload_file

login(token=userdata.get('HF_TOKEN'))

TU_USUARIO_HF = "santiagortegasegura"
REPO_DATOS = f"{TU_USUARIO_HF}/emotion-futbol-data"

RUTA_POOL_DRIVE = '/content/drive/MyDrive/RUTA/A/data_prep/pool_etiquetado.csv'
SEED = 42
UMBRAL_CONFIANZA = 0.67
N_PRUEBA = 10_000
EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']
COLUMNAS = ['comment_id', 'text', 'label', 'confianza']

def normalizar(texto):
    return " ".join(str(texto).lower().split())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 661.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 35.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.0.0 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.0.0 wh

## 2. Cargar el pool etiquetado y el split oficial

In [2]:
if os.path.exists(RUTA_POOL_DRIVE):
    ruta_pool = RUTA_POOL_DRIVE
elif os.path.exists('pool_etiquetado.csv'):
    ruta_pool = 'pool_etiquetado.csv'
else:
    print("Sube pool_etiquetado.csv:")
    ruta_pool = list(files.upload().keys())[0]

df_pool = pd.read_csv(ruta_pool)
df_pool = df_pool[(df_pool['label'] != 'no_valido') & (df_pool['confianza'] >= UMBRAL_CONFIANZA)].copy()
print(f"Pool etiquetado (mismo filtro de 04b_0): {len(df_pool):,}")

def bajar(nombre):
    return pd.read_csv(hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset", filename=nombre))

df_train = bajar("04b_entrenamiento_72k.csv")
df_val = bajar("04b_validacion_interna_18k.csv")
df_prueba_04b = bajar("04b_validacion_final_combinada.csv").reset_index(drop=True)
print(f"Entrenamiento: {len(df_train):,} | Validación interna: {len(df_val):,} | Prueba de 04b_0: {len(df_prueba_04b):,}")

ids_train_val = set(df_train['comment_id']) | set(df_val['comment_id'])
textos_train_val = set(df_train['text'].map(normalizar)) | set(df_val['text'].map(normalizar))

Sube pool_etiquetado.csv:


Saving pool_etiquetado.csv to pool_etiquetado.csv
Pool etiquetado (mismo filtro de 04b_0): 100,858


04b_entrenamiento_72k.csv:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

04b_validacion_interna_18k.csv:   0%|          | 0.00/2.16M [00:00<?, ?B/s]

04b_validacion_final_combinada.csv:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

Entrenamiento: 72,000 | Validación interna: 18,000 | Prueba de 04b_0: 10,000


## 3. Comentarios del conjunto preliminar (notebook 04) que se conservan

In [3]:
es_etiquetado_masivo = df_prueba_04b['origen'] == 'pool_llm'
textos_prueba = df_prueba_04b['text'].map(normalizar)

cruza_train_val = textos_prueba.isin(textos_train_val) | df_prueba_04b['comment_id'].isin(ids_train_val)
repetido_en_prueba = textos_prueba.duplicated(keep='first')

se_conserva = es_etiquetado_masivo & ~cruza_train_val & ~repetido_en_prueba
print(f"Comentarios del etiquetado masivo en la prueba de 04b_0: {es_etiquetado_masivo.sum():,}")
print(f"  que aparecen en entrenamiento/validación con otra escritura: {(es_etiquetado_masivo & cruza_train_val).sum()}")
print(f"  repetidos dentro de la misma prueba: {(es_etiquetado_masivo & repetido_en_prueba & ~cruza_train_val).sum()}")
print(f"Se conservan: {se_conserva.sum():,}")

df_conservados = df_prueba_04b[se_conserva].copy()
df_conservados['indice_04b'] = df_conservados.index
n_faltantes = N_PRUEBA - len(df_conservados)
print(f"Faltan para completar 10.000: {n_faltantes}")

Comentarios del etiquetado masivo en la prueba de 04b_0: 9,360
  que aparecen en entrenamiento/validación con otra escritura: 49
  repetidos dentro de la misma prueba: 0
Se conservan: 9,311
Faltan para completar 10.000: 689


## 4. Comentarios del etiquetado masivo que no se usaron en ninguna parte

In [4]:
ids_usados = ids_train_val | set(df_prueba_04b['comment_id'])
textos_usados = textos_train_val | set(textos_prueba)

df_disponibles = df_pool[~df_pool['comment_id'].isin(ids_usados)].copy()
df_disponibles = df_disponibles[~df_disponibles['text'].map(normalizar).isin(textos_usados)]
df_disponibles = df_disponibles[~df_disponibles['text'].map(normalizar).duplicated(keep='first')]

print(f"Disponibles sin usar: {len(df_disponibles):,}")
print(df_disponibles['label'].value_counts())
assert len(df_disponibles) >= n_faltantes, "No hay suficientes comentarios sin usar."

try:
    df_nuevos, _ = train_test_split(df_disponibles, train_size=n_faltantes,
                                    stratify=df_disponibles['label'], random_state=SEED)
except ValueError:
    df_nuevos = df_disponibles.sample(n=n_faltantes, random_state=SEED)
df_nuevos = df_nuevos.copy()
df_nuevos['indice_04b'] = -1
print(f"\nSeleccionados para completar: {len(df_nuevos)}")

Disponibles sin usar: 1,489
label
alegria     486
enojo       464
burla       244
neutral     159
tristeza     71
sorpresa     61
miedo         4
Name: count, dtype: int64

Seleccionados para completar: 689


## 5. Armar la prueba final y verificarla

In [5]:
df_prueba = pd.concat([df_conservados, df_nuevos], ignore_index=True)
textos_final = df_prueba['text'].map(normalizar)
ids_mano = set(df_prueba_04b.loc[~es_etiquetado_masivo, 'comment_id'])
textos_mano = set(textos_prueba[~es_etiquetado_masivo])

chequeos = {
    "exactamente 10.000 comentarios": len(df_prueba) == N_PRUEBA,
    "ningún id repetido": df_prueba['comment_id'].is_unique,
    "ningún texto repetido (ignorando mayúsculas y espacios)": textos_final.is_unique,
    "nada en entrenamiento/validación (id)": not df_prueba['comment_id'].isin(ids_train_val).any(),
    "nada en entrenamiento/validación (texto)": not textos_final.isin(textos_train_val).any(),
    "nada de la muestra etiquetada a mano": not (df_prueba['comment_id'].isin(ids_mano).any()
                                                   or textos_final.isin(textos_mano).any()),
    "todas las etiquetas son de las 7 clases": df_prueba['label'].isin(EMOCIONES).all(),
    "confianza del etiquetado >= 0.67": (df_prueba['confianza'] >= UMBRAL_CONFIANZA).all(),
}
for nombre, ok in chequeos.items():
    print(f"[{'OK' if ok else 'FALLA'}] {nombre}")
assert all(chequeos.values()), "Alguna verificación falló -- no se guarda nada."

print("\nDistribución de emociones en la prueba final:")
print(df_prueba['label'].value_counts().reindex(EMOCIONES))

[OK] exactamente 10.000 comentarios
[OK] ningún id repetido
[OK] ningún texto repetido (ignorando mayúsculas y espacios)
[OK] nada en entrenamiento/validación (id)
[OK] nada en entrenamiento/validación (texto)
[OK] nada de la muestra etiquetada a mano
[OK] todas las etiquetas son de las 7 clases
[OK] confianza del etiquetado >= 0.67

Distribución de emociones en la prueba final:
label
alegria     3282
tristeza     479
enojo       3104
miedo         26
sorpresa     409
burla       1633
neutral     1067
Name: count, dtype: int64


## 6. Guardar en Hugging Face

In [6]:
df_prueba[COLUMNAS].to_csv("prueba_final_10k.csv", index=False)
df_prueba[['comment_id', 'indice_04b']].to_csv("mapa_prueba_final.csv", index=False)

resumen = {
    "n_total": len(df_prueba),
    "n_conservados_de_04b": int((df_prueba['indice_04b'] >= 0).sum()),
    "n_nuevos": int((df_prueba['indice_04b'] < 0).sum()),
    "semilla": SEED,
    "distribucion": df_prueba['label'].value_counts().to_dict(),
    "fecha": pd.Timestamp.now().isoformat(),
}
with open("resumen_prueba_final.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

for local, remoto in [("prueba_final_10k.csv", "final/prueba_final_10k.csv"),
                      ("mapa_prueba_final.csv", "final/auxiliar/mapa_prueba_final.csv"),
                      ("resumen_prueba_final.json", "final/resumen_prueba_final.json"),
                      (ruta_pool, "datos/pool_etiquetado.csv")]:
    upload_file(path_or_fileobj=local, path_in_repo=remoto, repo_id=REPO_DATOS, repo_type="dataset")
    print(f"Subido: {remoto}")

print(json.dumps(resumen, indent=2, ensure_ascii=False))

Subido: final/prueba_final_10k.csv
Subido: final/auxiliar/mapa_prueba_final.csv
Subido: final/resumen_prueba_final.json


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/pool_etiquetado.csv:   6%|5         |  771kB / 12.9MB            

Validating                    :           |   0%            

Subido: datos/pool_etiquetado.csv
{
  "n_total": 10000,
  "n_conservados_de_04b": 9311,
  "n_nuevos": 689,
  "semilla": 42,
  "distribucion": {
    "alegria": 3282,
    "enojo": 3104,
    "burla": 1633,
    "neutral": 1067,
    "tristeza": 479,
    "sorpresa": 409,
    "miedo": 26
  },
  "fecha": "2026-09-25T16:01:46.693134"
}
